In [39]:
# -----######-----###### PLAYLIST → MP3 (PNG cover) + RANK + NAME-CODE + RB TXT/M3U8 -----######-----###### #
import os, sys, shlex, subprocess, hashlib, shutil, math, re
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Tag libs
from mutagen import File as MutaFile
from mutagen.id3 import ID3, ID3NoHeaderError, ID3BadUnsynchData
from mutagen.id3 import TIT2, TPE1, TPE2, TALB, TRCK, TCON, TDRC, TKEY, TBPM, TPE4, TPUB, TXXX, COMM, APIC
from mutagen.easyid3 import EasyID3
from mutagen.mp3 import MP3

# ===== helpers (no ASCII banner for sub-fns) =====
def _md5_of_file(path, chunk_size=1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def _which_ffmpeg():
    return shutil.which("ffmpeg")

def _safe_new_path(dest_folder, base_stem, ext=".mp3"):
    """
    Return a non-colliding path like:
      dest/fname.mp3, dest/fname (1).mp3, dest/fname (2).mp3, ...
    """
    dest_folder = Path(dest_folder)
    p = dest_folder / f"{base_stem}{ext}"
    if not p.exists():
        return str(p)
    i = 1
    while True:
        q = dest_folder / f"{base_stem} ({i}){ext}"
        if not q.exists():
            return str(q)
        i += 1

def _copy_basic_tags_to_id3(id3obj, easy):
    mapping = {
        "title": TIT2,
        "artist": TPE1,
        "album": TALB,
        "tracknumber": TRCK,
        "genre": TCON,
    }
    for k, frame in mapping.items():
        if k in easy and len(easy[k]) > 0:
            try:
                id3obj.setall(frame.__name__, [])
                id3obj.add(frame(encoding=3, text=easy[k][0]))
            except Exception:
                pass
    if "date" in easy and len(easy["date"]) > 0:
        try:
            id3obj.setall(TDRC.__name__, [])
            id3obj.add(TDRC(encoding=3, text=easy["date"][0]))
        except Exception:
            pass
    if "comment" in easy and len(easy["comment"]) > 0:
        try:
            id3obj.setall(COMM.__name__, [])
            id3obj.add(COMM(encoding=3, lang='eng', desc='', text=easy["comment"][0]))
        except Exception:
            pass

def _pull_extra_source_tags(src, id3obj):
    try:
        if not src:
            return
        pairs = [
            ("bpm", TBPM), ("TBPM", TBPM),
            ("key", TKEY), ("TKEY", TKEY),
            ("publisher", TPUB), ("TPUB", TPUB),
            ("remixer", TPE4), ("TPE4", TPE4),
        ]
        all_keys = set()
        try:
            all_keys.update([k for k in src.keys()])
        except Exception:
            pass

        def get_first(keys):
            for k in keys:
                if k in src and src[k]:
                    v = src[k]
                    if isinstance(v, (list, tuple)) and len(v) > 0:
                        return str(v[0])
                    return str(v)
            for k in keys:
                for kk in all_keys:
                    if kk.lower() == k.lower():
                        v = src.get(kk)
                        if isinstance(v, (list, tuple)) and len(v) > 0:
                            return str(v[0])
                        return str(v)
            return None

        for k, Frame in pairs:
            val = get_first([k])
            if val:
                try:
                    id3obj.setall(Frame.__name__, [])
                    id3obj.add(Frame(encoding=3, text=str(val)))
                except Exception:
                    pass
    except Exception:
        pass

def _embed_png_cover(id3obj, png_path):
    with open(png_path, "rb") as f:
        id3obj.setall(APIC.__name__, [])
        id3obj.add(APIC(
            encoding=3,
            mime="image/png",
            type=3,
            desc="cover",
            data=f.read()
        ))

def _convert_to_mp3_ffmpeg(src_path, out_path):
    cmd = [
        _which_ffmpeg(), "-y",
        "-i", src_path,
        "-vn",
        "-ar", "44100",
        "-ac", "2",
        "-b:a", "320k",
        out_path
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"ffmpeg error: {p.stderr.strip()[:4000]}")

def _verify_mp3_ok(path):
    try:
        m = MP3(path)
        return (m.info is not None) and (getattr(m.info, "length", 0) > 0)
    except Exception:
        return False

def _rank_prefix(idx, total, pad=False, pad_width=None):
    if pad:
        width = pad_width or max(2, len(str(total)))
        return f"{str(idx).zfill(width)}"
    return f"{idx}"

def _slugify_name(stem):
    s = re.sub(r"[^\w\s\-\.]+", "", stem)
    s = re.sub(r"\s+", "_", s.strip())
    return s

def _name_code(stem, mode="hash", length=6):
    """
    Derive a short code from the song name (stem).
      - hash: md5(stem) -> first N chars
      - slug: first letters of up to N words (fallback to hash if empty)
    """
    if mode == "slug":
        parts = re.findall(r"[A-Za-z0-9]+", stem)
        if parts:
            code = "".join(p[0] for p in parts)[:length].upper()
            if code:
                return code
    # default: hash
    return hashlib.md5(stem.encode("utf-8")).hexdigest()[:length].upper()

def _build_ranked_stem(rank_num, stem_orig, total, keyword="", code_mode="hash", code_len=6, pad=False, pad_width=None):
    rank_str = _rank_prefix(rank_num, total, pad=pad, pad_width=pad_width)
    code = _name_code(stem_orig, mode=code_mode, length=code_len)
    kw = (keyword.strip() + "_") if keyword else ""
    # Final pattern: {rank}_{kw}{code}_{original-stem}
    return f"{rank_str}_{kw}{code}_{_slugify_name(stem_orig)}"

def _write_rekordbox_txt(txt_path, rows):
    """
    Write a UTF-16 tab-separated TXT with at least Location + Name columns.
    rows: list of dicts with keys Location, Name
    """
    df = pd.DataFrame(rows)
    # Ensure required columns exist
    if "Location" not in df.columns: df["Location"] = ""
    if "Name" not in df.columns: df["Name"] = ""
    df.to_csv(txt_path, sep="\t", encoding="utf-16", index=False)

def _write_m3u8(m3u8_path, paths):
    with open(m3u8_path, "w", encoding="utf-8") as f:
        f.write("#EXTM3U\n")
        for p in paths:
            f.write(str(p) + "\n")

# -----######-----###### CORE IMPORTABLE FUNCTION -----######-----###### #
def _playlist_1908_mp3cover_GET_df_log(
    txt_path,
    dest_parent_folder,
    cover_png_path,
    save_logs=True,
    # ranking + code controls
    rank_prefix=True,
    rank_pad=False,
    rank_pad_width=None,
    keyword="",           # e.g., "PUkw" -> filename: 1_PUkw_AB12CD_songname.mp3
    name_code_mode="hash",# "hash" or "slug"
    name_code_len=6
):
    """
    Read a UTF-16 tab-separated TXT with 'Location'.
    Create output folder named exactly like the playlist file (no extension) under dest_parent_folder.
    Convert each audio (AIFF/WAV/FLAC/MP3) to MP3 (CBR 320k) into that folder,
    preserve tags, embed provided PNG cover, and prefix filenames by playlist order,
    then add a code derived from each song name (plus optional keyword before the code).

    Also exports in the SAME output folder:
      - playlist_<PLAYLISTNAME>.m3u8
      - playlist_<PLAYLISTNAME>.txt  (UTF-16 TSV with Location + Name)

    Returns
    -------
    DataFrame with: Rank, Original_Path, Output_MP3, Final_Name, status, msg, md5_out
    """
    # ===== TQM init =====
    tqdm.pandas()

    # Validations
    ffmpeg_bin = _which_ffmpeg()
    if not ffmpeg_bin:
        raise EnvironmentError("ffmpeg not found in PATH. Please install ffmpeg and try again.")
    cover_png_path = str(cover_png_path)
    if not os.path.isfile(cover_png_path):
        raise FileNotFoundError(f"cover_png_path not found: {cover_png_path}")

    txt_path = str(txt_path)
    if not os.path.isfile(txt_path):
        raise FileNotFoundError(f"txt_path not found: {txt_path}")

    # Derive output folder name from playlist file name (no extension)
    playlist_stem = Path(txt_path).stem
    dest_parent_folder = str(dest_parent_folder)
    dest_folder = str(Path(dest_parent_folder) / playlist_stem)
    os.makedirs(dest_folder, exist_ok=True)

    # Load playlist
    df = pd.read_csv(
        txt_path,
        sep="\t",
        encoding="utf-16",
        engine="python",
        on_bad_lines="skip"
    )
    df.columns = df.columns.str.strip()
    if 'Location' not in df.columns:
        raise ValueError("Playlist is missing required 'Location' column.")

    # Collect valid paths in order; missing logged later
    src_paths = []
    for _, row in df.iterrows():
        p = str(row.get('Location', '')).strip()
        if p and os.path.isfile(p):
            src_paths.append(p)

    total_valid = len(src_paths)
    base_dir = os.path.dirname(os.path.abspath(txt_path))
    records = []
    rb_rows = []       # for Rekordbox TXT export
    m3u_paths = []     # for M3U8 export

    # ===== CONVERT + TAG (TQM) =====
    for rank_idx, src in enumerate(tqdm(src_paths, desc="Converting to MP3 (TQM)"), start=1):
        stem_orig = Path(src).stem
        # Build ranked+coded stem
        if rank_prefix:
            stem_ranked = _build_ranked_stem(
                rank_num=rank_idx,
                stem_orig=stem_orig,
                total=total_valid,
                keyword=keyword,
                code_mode=name_code_mode,
                code_len=name_code_len,
                pad=rank_pad,
                pad_width=rank_pad_width
            )
        else:
            stem_ranked = _slugify_name(stem_orig)

        out_path = _safe_new_path(dest_folder, stem_ranked, ext=".mp3")
        final_name = Path(out_path).name

        try:
            # 1) Convert
            _convert_to_mp3_ffmpeg(src, out_path)

            # 2) Tags
            try:
                src_tags_easy = MutaFile(src, easy=True)
            except Exception:
                src_tags_easy = None

            try:
                id3 = ID3(out_path)
            except (ID3NoHeaderError, ID3BadUnsynchData):
                id3 = ID3()

            if src_tags_easy is not None:
                _copy_basic_tags_to_id3(id3, src_tags_easy)

            try:
                src_raw = MutaFile(src, easy=False)
            except Exception:
                src_raw = None
            if src_raw is not None:
                _pull_extra_source_tags(src_raw, id3)

            _embed_png_cover(id3, cover_png_path)
            id3.save(out_path, v2_version=3)

            # 3) Verify
            ok = _verify_mp3_ok(out_path)
            if not ok:
                raise RuntimeError("Output MP3 failed verification (unreadable or zero length).")

            md5o = _md5_of_file(out_path)
            records.append({
                "Rank": rank_idx,
                "Original_Path": src,
                "Output_MP3": out_path,
                "Final_Name": final_name,
                "status": "ok",
                "msg": "",
                "md5_out": md5o
            })

            # For RB exports
            rb_rows.append({"Location": out_path, "Name": Path(out_path).stem})
            m3u_paths.append(out_path)

        except Exception as e:
            try:
                if out_path and os.path.exists(out_path) and os.path.getsize(out_path) == 0:
                    os.remove(out_path)
            except Exception:
                pass
            records.append({
                "Rank": rank_idx,
                "Original_Path": src,
                "Output_MP3": None,
                "Final_Name": None,
                "status": "error",
                "msg": str(e),
                "md5_out": None
            })

    # Track missing sources for transparency (no rank)
    for _, row in df.iterrows():
        p = str(row.get('Location', '')).strip()
        if (not p) or (not os.path.isfile(p)):
            records.append({
                "Rank": None,
                "Original_Path": p,
                "Output_MP3": None,
                "Final_Name": None,
                "status": "missing",
                "msg": "Source file not found",
                "md5_out": None
            })

    report = pd.DataFrame.from_records(records).sort_values(
        by=["Rank", "status"], ascending=[True, True], na_position="last"
    ).reset_index(drop=True)

    # ===== Export Rekordbox playlist files inside the SAME playlist-named folder =====
    playlist_txt_out = str(Path(dest_folder) / f"playlist_{playlist_stem}.txt")
    playlist_m3u8_out = str(Path(dest_folder) / f"playlist_{playlist_stem}.m3u8")

    if rb_rows:
        _write_rekordbox_txt(playlist_txt_out, rb_rows)
        _write_m3u8(playlist_m3u8_out, m3u_paths)

    # ===== Save logs =====
    if save_logs:
        report.to_csv(str(Path(dest_folder) / "mp3_cover_report.csv"), index=False)
        if (report['status'] == 'error').any():
            report.loc[report['status'].eq('error')].to_csv(str(Path(dest_folder) / "mp3_cover_errors.csv"), index=False)
        if (report['status'] == 'missing').any():
            report.loc[report['status'].eq('missing'), ['Original_Path']].rename(columns={'Original_Path': 'Missing_Files'}).to_csv(str(Path(dest_folder) / "missing_files_log.csv"), index=False)

    # ===== Summary =====
    ok_n = int((report['status'] == 'ok').sum())
    err_n = int((report['status'] == 'error').sum())
    miss_n = int((report['status'] == 'missing').sum())
    print(f"\n✅ Done. OK: {ok_n} | Errors: {err_n} | Missing: {miss_n}")
    print(f"📁 Output folder: {dest_folder}")
    if rb_rows:
        print(f"📝 Rekordbox TXT: {playlist_txt_out}")
        print(f"🎵 M3U8: {playlist_m3u8_out}")
    return report


In [40]:
# EDIT THESE
txt_path = "/Users/yerik/Downloads/2c_LATINFIRE_mina.txt"          # UTF-16 TSV with a 'Location' column
dest_parent_folder = "/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs"       # Where MP3s go
cover_png_path = "/Users/yerik/_apple_source/all_data/GLOBAL/_2_jpegs_pngs_mp4_sys/_3_cover_images/_2bbs_.png"            # One PNG cover applied to ALL tracks

# RANK + CODE CONTROLS
rank_prefix = True          # prefix filenames as 1_, 2_, ...
rank_pad = False            # set True to get 01_, 02_, ... (auto width)
rank_pad_width = None       # or set an int (e.g., 3 → 001_, 002_)
keyword = "PUkw"            # optional keyword inserted before the code ("" to disable)
name_code_mode = "hash"     # "hash" (stable 6-char) or "slug" (acronym from words)
name_code_len = 6           # code length (e.g., 6)

# RUN
report = _playlist_1908_mp3cover_GET_df_log(
    txt_path=txt_path,
    dest_parent_folder=dest_parent_folder,
    cover_png_path=cover_png_path,
    save_logs=True,
    rank_prefix=rank_prefix,
    rank_pad=rank_pad,
    rank_pad_width=rank_pad_width,
    keyword=keyword,
    name_code_mode=name_code_mode,
    name_code_len=name_code_len
)

# report columns: Rank, Original_Path, Output_MP3, Final_Name, status, msg, md5_out


Converting to MP3 (TQM): 100%|██████████████████████████████████████████████████| 2/2 [00:04<00:00,  2.20s/it]


✅ Done. OK: 2 | Errors: 0 | Missing: 0
📁 Output folder: /Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina
📝 Rekordbox TXT: /Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/playlist_2c_LATINFIRE_mina.txt
🎵 M3U8: /Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/playlist_2c_LATINFIRE_mina.m3u8


# if 2bbs 

In [41]:
_2bb_folder = "/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina"

In [42]:
# end

In [43]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Rekordbox-Grid Safe Same-BPM, In-Place MP3) -----######-----###### #
import os, math, shutil, tempfile, subprocess, datetime
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm

from mutagen.id3 import ID3, ID3NoHeaderError, TBPM, TXXX
from mutagen.mp3 import MP3

# ---------- helpers (no ASCII art) ----------

def _which(cmd):
    from shutil import which
    return which(cmd)

def _rb_cmd():
    """Prefer rubberband-r3 (higher quality). Fallback to rubberband. Return (cmd, engine_label)."""
    rb3 = _which("rubberband-r3")
    if rb3:
        return rb3, "rubberband-r3"
    rb = _which("rubberband")
    if rb:
        return rb, "rubberband"
    return None, None

def _ensure_ffmpeg():
    if _which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install via Homebrew: brew install ffmpeg")

def _load_audio_stereo(path, target_sr=None):
    y, sr = librosa.load(path, sr=target_sr, mono=False)
    if y.ndim == 1:
        y = np.expand_dims(y, 0)
    return y, sr  # (ch, n)

def _mono(y):
    return np.mean(y, axis=0) if y.ndim == 2 else y

def _estimate_bpm_and_beats(y, sr, hop_length=512):
    odf = librosa.onset.onset_strength(y=_mono(y), sr=sr, hop_length=hop_length)
    tempo, beats = librosa.beat.beat_track(onset_envelope=odf, sr=sr, hop_length=hop_length, trim=True)
    tempo = float(np.atleast_1d(tempo)[0])  # fix deprecation warning / ensure scalar
    return tempo, beats, hop_length

def _frames_to_samples(frames, hop_length):
    return librosa.frames_to_samples(frames, hop_length=hop_length)

def _choose_first_downbeat(beats, hop_length):
    # Try 4/4 downbeat: first beat where index%4==0; fallback to first beat.
    if beats is None or len(beats) == 0:
        return 0
    for k, f in enumerate(beats):
        if k % 4 == 0:
            return int(_frames_to_samples(f, hop_length))
    return int(_frames_to_samples(beats[0], hop_length))

def _db_to_amp(db):
    return 10.0 ** (db / 20.0)

def _trim_tail(y, sr, thresh_db=-42, pad_ms=20):
    amp_thr = _db_to_amp(thresh_db)
    y_m = _mono(y)
    hits = np.where(np.abs(y_m) > amp_thr)[0]
    if hits.size == 0:
        cut = 0
    else:
        last_strong = int(hits[-1])
        cut = min(len(y_m), last_strong + int(sr * (pad_ms/1000.0)))
    return y[:, :cut] if y.ndim == 2 else y[:cut]

def _fade_edges(y, sr, fade_ms=10):
    n = y.shape[-1]
    n_fade = max(1, int(sr * (fade_ms / 1000.0)))
    if n < 2 * n_fade:
        return y
    if y.ndim == 1:
        y = y.reshape(1, -1)
    out = y.copy()
    ramp_in = np.linspace(0.0, 1.0, n_fade, dtype=np.float32)
    ramp_out = np.linspace(1.0, 0.0, n_fade, dtype=np.float32)
    for c in range(out.shape[0]):
        out[c, :n_fade] *= ramp_in
        out[c, -n_fade:] *= ramp_out
    return out

def _export_mp3_from_wav(tmp_wav, out_mp3, bitrate="320k"):
    _ensure_ffmpeg()
    cmd = ["ffmpeg","-y","-i",str(tmp_wav),"-vn","-codec:a","libmp3lame","-b:a",bitrate,str(out_mp3)]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"ffmpeg encode failed: {p.stderr.strip()}")

def _copy_all_id3_cover_and_add_bpm(src_mp3, dst_mp3, bpm_override=None, orig_bpm=None):
    try:
        tags = ID3(src_mp3)
    except ID3NoHeaderError:
        tags = None
    if tags is not None:
        tags.save(dst_mp3)
    try:
        new_tags = ID3(dst_mp3)
    except ID3NoHeaderError:
        new_tags = ID3()
    if bpm_override is not None:
        new_tags["TBPM"] = TBPM(encoding=3, text=str(int(round(bpm_override))))
    if orig_bpm is not None:
        new_tags.add(TXXX(encoding=3, desc="ORIG_BPM", text=str(round(orig_bpm, 3))))
    new_tags.save(dst_mp3)

def _atomic_replace(src_path, dst_path):
    backup = Path(str(src_path) + ".bak_tmp")
    shutil.move(str(src_path), backup)
    try:
        shutil.move(str(dst_path), str(src_path))
        backup.unlink(missing_ok=True)
    except Exception as e:
        shutil.move(str(backup), str(src_path))
        raise e

def _rubberband_stretch_wav(in_wav, out_wav, rate, formant=True):
    """
    Correct CLI: rubberband [options] infile.wav outfile.wav
      - Use rubberband-r3 if available for better quality.
      - -t <ratio> : tempo ratio (rate)
      - -F         : formant preservation (safe even if not pitch-shifting)
      - --quiet    : suppress progress output
      - If using "rubberband" (R2 engine), set -c 0 (neutral crispness).
    """
    rb, label = _rb_cmd()
    if not rb:
        raise RuntimeError("Rubber Band CLI not found. Install: brew install rubberband")
    args = [rb, "--quiet", "-t", f"{rate:.8f}"]
    if formant:
        args.append("-F")
    if label == "rubberband":  # R2 engine crispness flag is valid here
        args += ["-c", "0"]
    # NOTE: correct order: infile then outfile (no -o)
    args += [str(in_wav), str(out_wav)]
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"rubberband failed: {p.stderr.strip()}")

def _librosa_time_stretch_stereo(y, rate):
    if abs(rate - 1.0) < 1e-8:
        return y
    chans = [librosa.effects.time_stretch(y[c], rate) for c in range(y.shape[0])]
    maxlen = max(len(ch) for ch in chans)
    out = np.zeros((len(chans), maxlen), dtype=np.float32)
    for i, ch in enumerate(chans):
        out[i, :len(ch)] = ch
    return out

def _to_rekordbox_uri(p: Path) -> str:
    # Rekordbox expects file://localhost/… URIs.
    return "file://localhost" + p.resolve().as_posix()

def _write_rekordbox_xml(xml_path, track_infos, product_name="YerikoTools", product_version="1.0"):
    """
    track_infos = list of dicts with:
      - track_id, name, artist, duration, avg_bpm, location_uri, tempo_start, tempo_bpm, metro, battito
    Writes a minimal COLLECTION-only rekordbox.xml with a constant BeatGrid (TEMPO).
    """
    from xml.sax.saxutils import escape
    lines = []
    lines.append('<?xml version="1.0" encoding="UTF-8"?>')
    lines.append('<DJ_PLAYLISTS Version="1,0,0">')
    lines.append(f'  <PRODUCT Name="{escape(product_name)}" Version="{escape(product_version)}" Company="YODJ"/>')
    lines.append(f'  <COLLECTION Entries="{len(track_infos)}">')
    for ti in track_infos:
        name = escape(ti.get("name",""))
        artist = escape(ti.get("artist",""))
        loc = escape(ti["location_uri"])
        dur = f'{ti.get("duration",0.0):.3f}'
        abpm = f'{ti.get("avg_bpm", ti["tempo_bpm"]):.3f}'
        lines.append(f'    <TRACK TrackID="{ti["track_id"]}" Name="{name}" Artist="{artist}" TotalTime="{dur}" AverageBpm="{abpm}" Location="{loc}">')
        lines.append(f'      <TEMPO Inizio="{ti.get("tempo_start",0.0):.6f}" Bpm="{ti["tempo_bpm"]:.6f}" Metro="{ti.get("metro","4/4")}" Battito="{ti.get("battito",1)}"/>')
        lines.append(f'      <POSITION_MARK Name="BEAT-1" Type="3" Start="{ti.get("tempo_start",0.0):.6f}" End="{ti.get("tempo_start",0.0):.6f}" Num="-1"/>')
        lines.append('    </TRACK>')
    lines.append('  </COLLECTION>')
    lines.append('  <PLAYLISTS>')
    lines.append('    <NODE Type="0" Name="ROOT" Count="0"/>')
    lines.append('  </PLAYLISTS>')
    lines.append('</DJ_PLAYLISTS>')
    Path(xml_path).write_text("\n".join(lines), encoding="utf-8")
    return str(xml_path)

# -----######-----###### MAIN IMPORTABLE (ASCII banner) -----######-----###### #
def _bpm_1908_rbxgrid_GET_inplace_samebpm(
    inputs,
    target_bpm,
    delta_limit=4.0,
    force=False,
    align_downbeat=True,
    trim_tail_db=-42,
    fade_ms=10,
    resample_sr=44100,
    write_bitrate="320k",
    prefer_rubberband=True,
    update_tbpm=True,
    emit_rekordbox_xml=True,
    product_name="YerikoTools",
    product_version="1.0",
    verbose=True
):
    """
    Detect BPM & beats, cut to downbeat (bar 1), trim tail, add micro-fades,
    time-stretch uniformly to EXACT target_bpm, preserve all ID3 + cover,
    update TBPM (+ store TXXX:ORIG_BPM), and optionally emit rekordbox.xml
    with a constant BeatGrid starting at 0.000s.
    """
    try:
        import pandas as pd
        is_df = hasattr(inputs, "to_frame") or isinstance(inputs, pd.DataFrame)
    except Exception:
        is_df = False

    paths = list(inputs["Path"]) if is_df else [str(p) for p in inputs]
    if len(paths) == 0:
        raise ValueError("No input files provided.")
    for p in paths:
        if not str(p).lower().endswith(".mp3"):
            raise ValueError(f"Not an MP3: {p}")

    rb_cmd, rb_label = _rb_cmd()
    use_rb = prefer_rubberband and (rb_cmd is not None)
    if verbose:
        print(f"TQM • Target BPM: {float(target_bpm):.3f}")
        print(f"TQM • Engine: {rb_label if use_rb else 'Librosa'}")

    # Stage 1: detect
    det, skipped = [], []
    if verbose:
        print("TQM • Detecting BPM & beats…")
    for p in tqdm(paths, desc="Detect", unit="file"):
        y, sr = _load_audio_stereo(p, target_sr=resample_sr)
        tempo, beats, hop = _estimate_bpm_and_beats(y, sr, hop_length=512)
        fb = _choose_first_downbeat(beats, hop) if align_downbeat else (0 if beats is None or len(beats)==0 else int(_frames_to_samples(beats[0], hop)))
        reason = ""
        if (abs(float(target_bpm) - tempo) > delta_limit) and not force:
            reason = f"ΔBPM {abs(float(target_bpm)-tempo):.2f} > limit {delta_limit}"
        det.append({"path": p, "sr": sr, "bpm_est": float(tempo), "first_beat": int(fb), "skip_reason": reason})

    # Stage 2: process
    results = []
    if verbose:
        print("TQM • Processing in-place…")
    for meta in tqdm(det, desc="Process", unit="file"):
        src = Path(meta["path"])
        if meta["skip_reason"]:
            skipped.append({"path": str(src), "bpm_est": meta["bpm_est"], "reason": meta["skip_reason"]})
            continue

        y, sr = _load_audio_stereo(src, target_sr=resample_sr)
        # Cut/trim/fade
        y = y[:, meta["first_beat"]:] if y.ndim == 2 else y[meta["first_beat"]:]
        y = _trim_tail(y, sr, thresh_db=trim_tail_db)
        y = _fade_edges(y, sr, fade_ms=fade_ms)

        # Rate
        est = max(1e-6, float(meta["bpm_est"]))
        rate = float(target_bpm) / est

        with tempfile.TemporaryDirectory() as td:
            tmp_in = Path(td)/"in.wav"
            tmp_out = Path(td)/"out.wav"
            tmp_mp3 = Path(td)/"out.mp3"
            y2 = y if y.ndim == 2 else np.vstack([y, y])
            sf.write(str(tmp_in), y2.T, sr, subtype="PCM_24")

            if use_rb and abs(rate - 1.0) > 1e-8:
                _rubberband_stretch_wav(tmp_in, tmp_out, rate=rate, formant=True)
            elif use_rb:
                shutil.copyfile(tmp_in, tmp_out)
            else:
                y_st = _librosa_time_stretch_stereo(y2, rate) if abs(rate - 1.0) > 1e-8 else y2
                sf.write(str(tmp_out), y_st.T, sr, subtype="PCM_24")

            _export_mp3_from_wav(tmp_out, tmp_mp3, bitrate=write_bitrate)
            _copy_all_id3_cover_and_add_bpm(str(src), str(tmp_mp3),
                                            bpm_override=float(target_bpm) if update_tbpm else None,
                                            orig_bpm=meta["bpm_est"])
            _atomic_replace(str(src), str(tmp_mp3))

        # duration for XML
        try:
            dur_s = float(MP3(str(src)).info.length)
        except Exception:
            dur_s = (y.shape[-1] / sr) * (est/float(target_bpm))

        results.append({
            "path": str(src),
            "orig_bpm_est": float(meta["bpm_est"]),
            "new_bpm": float(target_bpm),
            "rate": float(rate),
            "start_cut_samples": int(meta["first_beat"]),
            "sr": int(sr),
            "engine": rb_label if use_rb else "librosa",
            "duration_s": dur_s
        })

    # Rekordbox XML (optional)
    rbx_path = ""
    if emit_rekordbox_xml and len(results) > 0:
        first_folder = Path(results[0]["path"]).parent
        rbx_path = str(first_folder / "rekordbox.xml")
        track_infos = []
        for i, r in enumerate(results, start=1):
            p = Path(r["path"])
            track_infos.append({
                "track_id": i,
                "name": p.stem,
                "artist": "",
                "duration": r["duration_s"],
                "avg_bpm": r["new_bpm"],
                "location_uri": _to_rekordbox_uri(p),
                "tempo_start": 0.0,
                "tempo_bpm": r["new_bpm"],
                "metro": "4/4",
                "battito": 1
            })
        _write_rekordbox_xml(rbx_path, track_infos, product_name=product_name, product_version=product_version)

    # Return (df or dict)
    try:
        import pandas as pd
        if is_df:
            idx_map = {p:i for i,p in enumerate(paths)}
            for r in results:
                i = idx_map[r["path"]]
                inputs.loc[i, "bpm_est"] = r["orig_bpm_est"]
                inputs.loc[i, "bpm_new"] = r["new_bpm"]
                inputs.loc[i, "rate_applied"] = r["rate"]
                inputs.loc[i, "start_cut_samples"] = r["start_cut_samples"]
                inputs.loc[i, "sr_proc"] = r["sr"]
                inputs.loc[i, "engine"] = r["engine"]
                inputs.loc[i, "rbx_xml"] = rbx_path
            if len(skipped) > 0:
                sk_map = {s["path"]: s["reason"] for s in skipped}
                inputs["skip_reason"] = inputs["Path"].map(lambda p: sk_map.get(p, ""))
            inputs["Path_synced"] = inputs["Path"]
            return inputs
    except Exception:
        pass

    return {"processed": results, "skipped": skipped, "rekordbox_xml": rbx_path}
# -----######-----###### END CORE FUNCTION -----######-----###### #


In [44]:
from pathlib import Path

# ======= EDIT ME =======
folder = _2bb_folder
target_bpm = 128          # <-- set EXACT BPM you want
delta_limit = 4.0         # ±4 rail (set force=True to override)
force = False
prefer_rubberband = True  # uses rubberband-r3 if present, else rubberband; else falls back to Librosa
# =======================

mp3s = sorted([str(p) for p in Path(folder).glob("*.mp3")])
if len(mp3s) < 2:
    raise SystemExit(f"Need at least 2 mp3 files in {folder}")

summary = _bpm_1908_rbxgrid_GET_inplace_samebpm(
    inputs=mp3s[:2],                 # or mp3s to process all
    target_bpm=target_bpm,
    delta_limit=delta_limit,
    force=force,
    align_downbeat=True,
    trim_tail_db=-42,
    fade_ms=10,
    resample_sr=44100,
    write_bitrate="320k",
    prefer_rubberband=prefer_rubberband,
    update_tbpm=True,
    emit_rekordbox_xml=True,
    product_name="YerikoTools",
    product_version="1.0",
    verbose=True
)

print("\nTQM • Done.\n", summary)


TQM • Target BPM: 128.000
TQM • Engine: rubberband-r3
TQM • Detecting BPM & beats…


Detect: 100%|█████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.02file/s]


TQM • Processing in-place…


Process: 100%|████████████████████████████████████████████████████████████████| 2/2 [00:42<00:00, 21.40s/file]


TQM • Done.
 {'processed': [{'path': '/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/1_PUkw_EEDCBA_Constantine_-_Constantine_-_03_Essa_Mina.mp3', 'orig_bpm_est': 129.19921875, 'new_bpm': 128.0, 'rate': 0.9907180650037792, 'start_cut_samples': 20992, 'sr': 44100, 'engine': 'rubberband-r3', 'duration_s': 175.15102040816328}, {'path': '/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/2_PUkw_1F07FD_NUEVAYoL_Edit.mp3', 'orig_bpm_est': 129.19921875, 'new_bpm': 128.0, 'rate': 0.9907180650037792, 'start_cut_samples': 20992, 'sr': 44100, 'engine': 'rubberband-r3', 'duration_s': 413.36163265306124}], 'skipped': [], 'rekordbox_xml': '/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/rekordbox.xml'}


In [23]:
# -----######-----###### CORE IMPORTABLE FUNCTION (On-Beat, Safe Sync to Target BPM — In-Place MP3) -----######-----###### #
import os, sys, math, shutil, tempfile, subprocess
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm

from mutagen.id3 import ID3, ID3NoHeaderError, TBPM, TXXX
from mutagen.mp3 import MP3

# --------------------------- helpers (no ASCII art) ---------------------------

def _which(cmd):
    from shutil import which
    return which(cmd)

def _have_rubberband():
    return _which("rubberband") is not None

def _ensure_ffmpeg():
    if _which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found. Install: brew install ffmpeg")

def _load_audio_stereo(path, target_sr=None):
    y, sr = librosa.load(path, sr=target_sr, mono=False)
    if y.ndim == 1:
        y = np.expand_dims(y, 0)
    return y, sr  # (ch, n)

def _mono_for_detection(y):
    return np.mean(y, axis=0) if y.ndim == 2 else y

def _estimate_bpm_and_beats(y, sr, hop_length=512):
    y_mono = _mono_for_detection(y)
    odf = librosa.onset.onset_strength(y=y_mono, sr=sr, hop_length=hop_length)
    tempo, beats = librosa.beat.beat_track(onset_envelope=odf, sr=sr, hop_length=hop_length, trim=True)
    return float(tempo), beats

def _frames_to_samples(frames, hop_length=512):
    return librosa.frames_to_samples(frames, hop_length=hop_length)

def _first_beat_sample(beats, hop_length=512):
    if beats is None or len(beats) == 0:
        return 0
    return int(_frames_to_samples(beats[0], hop_length=hop_length))

def _db_to_amp(db):
    return 10.0 ** (db / 20.0)

def _trim_tail(y, sr, thresh_db=-42, min_silence_ms=300):
    amp_thr = _db_to_amp(thresh_db)
    y_mono = _mono_for_detection(y)
    abs_y = np.abs(y_mono)
    hits = np.where(abs_y > amp_thr)[0]
    if hits.size == 0:
        cut = 0
    else:
        last_strong = int(hits[-1])
        cut = min(len(abs_y), last_strong + int(sr * 0.02))  # +20ms margin
    return y[:, :cut] if y.ndim == 2 else y[:cut]

def _fade_edges(y, sr, fade_ms=10):
    n = y.shape[-1]
    n_fade = max(1, int(sr * (fade_ms / 1000.0)))
    if n < 2 * n_fade:
        return y
    if y.ndim == 1:
        y = y.reshape(1, -1)
    out = y.copy()
    ramp_in = np.linspace(0.0, 1.0, n_fade, dtype=np.float32)
    ramp_out = np.linspace(1.0, 0.0, n_fade, dtype=np.float32)
    for c in range(out.shape[0]):
        out[c, :n_fade] *= ramp_in
        out[c, -n_fade:] *= ramp_out
    return out

def _export_mp3_from_wav(tmp_wav, out_mp3, bitrate="320k"):
    _ensure_ffmpeg()
    cmd = ["ffmpeg","-y","-i",str(tmp_wav),"-vn","-codec:a","libmp3lame","-b:a",bitrate,str(out_mp3)]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"ffmpeg encode failed: {p.stderr.strip()}")

def _copy_all_id3_cover_and_add_bpm(src_mp3, dst_mp3, bpm_override=None, orig_bpm=None):
    try:
        tags = ID3(src_mp3)
    except ID3NoHeaderError:
        tags = None
    if tags is not None:
        tags.save(dst_mp3)

    # Ensure TBPM and ORIG_BPM are correct
    try:
        new_tags = ID3(dst_mp3)
    except ID3NoHeaderError:
        new_tags = ID3()
    if bpm_override is not None:
        new_tags["TBPM"] = TBPM(encoding=3, text=str(int(round(bpm_override))))
    if orig_bpm is not None:
        new_tags.add(TXXX(encoding=3, desc="ORIG_BPM", text=str(round(orig_bpm, 3))))
    new_tags.save(dst_mp3)

def _atomic_replace(src_path, dst_path):
    backup = Path(str(src_path) + ".bak_tmp")
    shutil.move(str(src_path), backup)
    try:
        shutil.move(str(dst_path), str(src_path))
        backup.unlink(missing_ok=True)
    except Exception as e:
        shutil.move(str(backup), str(src_path))
        raise e

def _rubberband_stretch_wav(in_wav, out_wav, rate, formant=True):
    # rate > 1 speeds up, < 1 slows down. Rubber Band expects tempo change in %
    # -t is tempo ratio; -T tempo percent; we’ll use -t
    args = ["rubberband", "-t", f"{rate:.8f}"]
    # high quality, crisp transients, formant preserve
    args += ["-c","0","-F"] if formant else ["-c","0"]
    # -c 0 = standard; could use HQ modes but this is already great
    args += ["-o", str(out_wav), str(in_wav)]
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"rubberband failed: {p.stderr.strip()}")

def _librosa_time_stretch_stereo(y, rate):
    if abs(rate - 1.0) < 1e-8:
        return y
    chans = []
    for c in range(y.shape[0]):
        chans.append(librosa.effects.time_stretch(y[c], rate))
    maxlen = max([len(ch) for ch in chans])
    out = np.zeros((len(chans), maxlen), dtype=np.float32)
    for i, ch in enumerate(chans):
        out[i, :len(ch)] = ch
    return out

# -----######-----###### CORE IMPORTABLE FUNCTION -----######-----###### #
def _bpm_1908_onbeat_GET_synced_mp3(
    inputs,
    target_bpm,
    delta_limit=4.0,
    force=False,
    trim_tail_db=-42,
    min_tail_silence_ms=300,
    fade_ms=10,
    resample_sr=44100,
    write_bitrate="320k",
    prefer_rubberband=True,
    update_tbpm=True,
    verbose=True
):
    """
    Input:
      - inputs: list of MP3 paths OR a DataFrame with column 'Path'
      - target_bpm: the BPM you want (float or int) — REQUIRED
      - delta_limit: max allowed |target_bpm - detected_bpm| in BPM (default 4.0)
      - force: if True, process even if delta exceeds limit (use with caution)
      - trim_tail_db: tail trim threshold in dBFS (e.g., -42)
      - min_tail_silence_ms: silence window helper for tail logic
      - fade_ms: edge fade duration
      - resample_sr: processing sample rate
      - write_bitrate: MP3 export bitrate
      - prefer_rubberband: if True, use Rubber Band CLI when available
      - update_tbpm: write TBPM tag to new BPM
      - verbose: print TQM info

    Behavior:
      - Snap to first beat, remove tail, micro-fades
      - Stretch/shrink to EXACT target_bpm (pitch preserved)
      - Enforce safety rail: |ΔBPM| <= delta_limit unless force=True
      - Preserve ALL ID3 + artwork; add TXXX:ORIG_BPM; update TBPM
      - In-place atomic replace
    Returns:
      - If DataFrame: same df with new columns appended
      - Else: list of dict summaries
    """

    try:
        import pandas as pd
        is_df = hasattr(inputs, "to_frame") or isinstance(inputs, pd.DataFrame)
    except Exception:
        is_df = False

    paths = list(inputs["Path"]) if is_df else [str(p) for p in inputs]
    if len(paths) == 0:
        raise ValueError("No input files provided.")
    for p in paths:
        if not str(p).lower().endswith(".mp3"):
            raise ValueError(f"Not an MP3: {p}")

    use_rb = prefer_rubberband and _have_rubberband()
    if verbose:
        print(f"TQM • Target BPM set to {float(target_bpm):.3f}")
        print(f"TQM • Time-stretch engine: {'Rubber Band' if use_rb else 'Librosa'}")
        if not use_rb and prefer_rubberband:
            print("TQM • Rubber Band not found; using Librosa fallback.")

    # Stage 1: detection
    det = []
    if verbose:
        print("TQM • Detecting BPM & first beat…")
    for p in tqdm(paths, desc="Detect", unit="file"):
        y, sr = _load_audio_stereo(p, target_sr=resample_sr)
        tempo, beats = _estimate_bpm_and_beats(y, sr, hop_length=512)
        if (abs(float(target_bpm) - tempo) > delta_limit) and not force:
            det.append({"path": p, "sr": sr, "bpm_est": float(tempo), "first_beat": _first_beat_sample(beats), "skip_reason": f"ΔBPM {abs(float(target_bpm)-tempo):.2f} > limit {delta_limit}"})
        else:
            det.append({"path": p, "sr": sr, "bpm_est": float(tempo), "first_beat": _first_beat_sample(beats), "skip_reason": ""})

    # Stage 2: process
    results, skipped = [], []
    if verbose:
        print("TQM • Processing (in-place)…")
    for meta in tqdm(det, desc="Process", unit="file"):
        src = Path(meta["path"])
        if meta.get("skip_reason"):
            skipped.append({"path": str(src), "bpm_est": meta["bpm_est"], "reason": meta["skip_reason"]})
            continue

        y, sr = _load_audio_stereo(src, target_sr=resample_sr)

        # snap to first beat, trim, fade
        fb = int(meta["first_beat"])
        y = y[:, fb:] if y.ndim == 2 else y[fb:]
        y = _trim_tail(y, sr, thresh_db=trim_tail_db, min_silence_ms=min_tail_silence_ms)
        y = _fade_edges(y, sr, fade_ms=fade_ms)

        # compute rate
        est = max(1e-6, float(meta["bpm_est"]))
        rate = float(target_bpm) / est

        with tempfile.TemporaryDirectory() as td:
            tmp_in_wav = Path(td) / "in.wav"
            tmp_out_wav = Path(td) / "out.wav"
            tmp_out_mp3 = Path(td) / "out.mp3"

            # ensure stereo
            y2 = y if y.ndim == 2 else np.vstack([y, y])
            sf.write(str(tmp_in_wav), y2.T, sr, subtype="PCM_24")

            # stretch
            if use_rb and abs(rate - 1.0) > 1e-8:
                _rubberband_stretch_wav(tmp_in_wav, tmp_out_wav, rate=rate, formant=True)
            elif use_rb and abs(rate - 1.0) <= 1e-8:
                shutil.copyfile(tmp_in_wav, tmp_out_wav)
            else:
                # Librosa fallback
                y_st = _librosa_time_stretch_stereo(y2, rate) if abs(rate - 1.0) > 1e-8 else y2
                sf.write(str(tmp_out_wav), y_st.T, sr, subtype="PCM_24")

            # encode to MP3
            _export_mp3_from_wav(tmp_out_wav, tmp_out_mp3, bitrate=write_bitrate)

            # carry tags + artwork; write TBPM & ORIG_BPM
            _copy_all_id3_cover_and_add_bpm(
                src_mp3=str(src),
                dst_mp3=str(tmp_out_mp3),
                bpm_override=float(target_bpm) if update_tbpm else None,
                orig_bpm=meta["bpm_est"]
            )

            # atomic replace
            _atomic_replace(src_path=str(src), dst_path=str(tmp_out_mp3))

        results.append({
            "path": str(src),
            "orig_bpm_est": float(meta["bpm_est"]),
            "new_bpm": float(target_bpm),
            "rate": float(rate),
            "start_cut_samples": int(meta["first_beat"]),
            "sr": int(sr),
            "engine": "rubberband" if use_rb else "librosa"
        })

    # Summarize
    try:
        import pandas as pd
        if is_df:
            m_ok = [r for r in results]
            if len(m_ok) > 0:
                idx_map = {p:i for i,p in enumerate(paths)}
                ok_idx = [idx_map[r["path"]] for r in m_ok]
                inputs.loc[ok_idx, "bpm_est"] = [r["orig_bpm_est"] for r in m_ok]
                inputs.loc[ok_idx, "bpm_new"] = [r["new_bpm"] for r in m_ok]
                inputs.loc[ok_idx, "rate_applied"] = [r["rate"] for r in m_ok]
                inputs.loc[ok_idx, "start_cut_samples"] = [r["start_cut_samples"] for r in m_ok]
                inputs.loc[ok_idx, "sr_proc"] = [r["sr"] for r in m_ok]
                inputs.loc[ok_idx, "engine"] = [r["engine"] for r in m_ok]
                inputs["Path_synced"] = inputs["Path"]  # in-place
            if len(skipped) > 0:
                # Optional: create a side column for skip reason
                sk_map = {s["path"]: s["reason"] for s in skipped}
                inputs["skip_reason"] = inputs["Path"].map(lambda p: sk_map.get(p, ""))
            return inputs
    except Exception:
        pass

    return {"processed": results, "skipped": skipped}
# -----######-----###### END CORE FUNCTION -----######-----###### #


In [24]:
from pathlib import Path

# ======= EDIT ME =======
folder = _2bb_folder
target_bpm = 123           # <<— YOU SET THIS (explicit)
delta_limit = 4.0           # safety rail ±4 BPM
force = False               # set True to override rail (not recommended)
prefer_rubberband = True    # uses Rubber Band CLI if installed
# =======================

mp3s = sorted([str(p) for p in Path(folder).glob("*.mp3")])
if len(mp3s) < 2:
    raise SystemExit(f"Need at least 2 mp3 files in {folder}")

summary = _bpm_1908_onbeat_GET_synced_mp3(
    inputs=mp3s[:2],           # or mp3s to process all
    target_bpm=target_bpm,
    delta_limit=delta_limit,
    force=force,
    trim_tail_db=-42,
    min_tail_silence_ms=300,
    fade_ms=10,
    resample_sr=44100,
    write_bitrate="320k",
    prefer_rubberband=prefer_rubberband,
    update_tbpm=True,
    verbose=True
)

print("\nTQM • Done.\n", summary)


TQM • Target BPM set to 123.000
TQM • Time-stretch engine: Rubber Band
TQM • Detecting BPM & first beat…


Detect:   0%|                                                                         | 0/2 [00:00<?, ?file/s]/var/folders/pp/n6z2gh0x56ngpvp6vx5ml9j00000gn/T/ipykernel_86768/2235827577.py:38: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(tempo), beats
Detect: 100%|█████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.03s/file]


TQM • Processing (in-place)…


Process: 100%|█████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 15621.24file/s]


TQM • Done.
 {'processed': [], 'skipped': [{'path': '/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/1_PUkw_EEDCBA_Constantine_-_Constantine_-_03_Essa_Mina.mp3', 'bpm_est': 129.19921875, 'reason': 'ΔBPM 6.20 > limit 4.0'}, {'path': '/Users/yerik/Music/_3_YODJ_ADDS-inbox/_2__2bbs/2c_LATINFIRE_mina/2_PUkw_1F07FD_NUEVAYoL_Edit.mp3', 'bpm_est': 129.19921875, 'reason': 'ΔBPM 6.20 > limit 4.0'}]}
